# 5. Spatial analysis: neighbourhoods, niches, interactions

Now we use the coordinates.
On one ANCA-GN core we ask which cell types sit together, group the tissue into recurring
**niches**, and look for **ligand-receptor** signalling between neighbouring types.

### Setup

In [ ]:
import sys; sys.path.insert(0, '../scripts')
import numpy as np, pandas as pd
import scanpy as sc, squidpy as sq
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import course_utils as cu
sc.settings.verbosity = 1
plt.rcParams.update({'figure.facecolor': 'white', 'figure.dpi': 110})

## 1. One sample and its spatial graph

> **Method note.** Spatial statistics are computed
per section on a neighbour graph. We use a Delaunay graph (follows tissue geometry). Every
result below is conditional on this graph and on this one core.

In [ ]:
adata = cu.load_annotated()
SAMPLE = 'X37'                      # one ANCA-GN core
sub = adata[adata.obs['sample'] == SAMPLE].copy()
sq.gr.spatial_neighbors(sub, coord_type='generic', delaunay=True)
print(sub.shape)

## 2. Neighbourhood enrichment

> **Method note.** Permute cell-type labels on the graph
and compare observed adjacency to the null (z-score). The diagonal is self-aggregation;
positive off-diagonal pairs co-locate (e.g. immune cells around injured tubules or vessels),
negative pairs avoid each other. It is relative to this core's composition.

In [ ]:
sq.gr.nhood_enrichment(sub, cluster_key='celltype', seed=0)
sq.pl.nhood_enrichment(sub, cluster_key='celltype', figsize=(7, 7), method='ward')

## 3. Niches

> **Method note: two routes to the same idea.** A **niche** is a recurring
local cell-type mixture. (A) The transparent route: summarise each cell's neighbourhood
composition and cluster it. (B) A **BANKSY-style** route: augment each cell's own expression
with its neighbours' mean expression, then cluster, which blends molecular state with
spatial context. Dedicated tools (BANKSY, CellCharter, nichePCA) formalise this. Compare the
two maps: they should agree on the major compartments (glomeruli, tubulo-interstitium,
immune foci).

In [ ]:
# (A) niches the transparent way: cluster each cell's neighbourhood cell-type composition
conn = sub.obsp['spatial_connectivities']
onehot = pd.get_dummies(sub.obs['celltype'])
comp = conn.dot(onehot.to_numpy(float)); comp /= comp.sum(1, keepdims=True).clip(min=1)
sub.obs['niche_comp'] = pd.Categorical(KMeans(6, n_init=10, random_state=0).fit_predict(comp).astype(str))
sq.pl.spatial_scatter(sub, color='niche_comp', shape=None, size=8, figsize=(6.5, 6.5))
pd.crosstab(sub.obs['niche_comp'], sub.obs['celltype'], normalize='index').round(2)

In [ ]:
# (B) niches with a BANKSY-style augmented feature: own profile + neighbour-mean profile
Xd = sub.X.toarray() if hasattr(sub.X, 'toarray') else np.asarray(sub.X)
conn_norm = conn.multiply(1 / conn.sum(1).A.clip(min=1))
nbr_mean = conn_norm.dot(Xd)
lam = 0.8
H = np.concatenate([Xd, lam * nbr_mean], axis=1)
bk = sc.AnnData(H, obs=sub.obs.copy())
sc.pp.pca(bk, n_comps=30); sc.pp.neighbors(bk, n_neighbors=15)
sc.tl.leiden(bk, resolution=0.5, flavor='igraph', n_iterations=2, directed=False)
sub.obs['niche_banksy'] = bk.obs['leiden'].values
sq.pl.spatial_scatter(sub, color='niche_banksy', shape=None, size=8, figsize=(6.5, 6.5))

## 4. Ligand-receptor interactions

> **Method note.** `squidpy.gr.ligrec` tests, for each
sender-receiver cell-type pair, whether a ligand and its receptor are co-expressed more than
chance (permutation). On a targeted panel only a handful of curated pairs are measurable, and
expression is a proxy for activity, so read hits as hypotheses (e.g. CXCL9/10/11 -> CXCR3
recruiting T cells). Segmentation spillover can move a ligand to the wrong cell (see the spillover note in
notebook 4; re-segmenting with Cellpose, Baysor, or Proseg reduces it).

In [ ]:
# ligand-receptor interactions between cell types (curated pairs in the panel)
cand = [('CXCL9','CXCR3'),('CXCL10','CXCR3'),('CXCL11','CXCR3'),('CCL2','CCR2'),('CCL5','CCR5'),
        ('CXCL12','CXCR4'),('CD40LG','CD40'),('TNF','TNFRSF1A'),('SELE','SELPLG'),('ICAM1','ITGAL'),
        ('VCAM1','ITGA4'),('PDGFB','PDGFRB'),('VEGFA','FLT1'),('VEGFA','KDR'),('C3','C5AR1')]
v = set(sub.var_names)
pairs = pd.DataFrame([(l, r) for l, r in cand if l in v and r in v], columns=['source', 'target'])
print(f'{len(pairs)} ligand-receptor pairs in the panel')
sq.gr.ligrec(sub, cluster_key='celltype', interactions=pairs, use_raw=False, n_perms=100, seed=0)
groups = [g for g in ['IMM','EC','PT','FIB','VSM/P','TAL'] if g in sub.obs['celltype'].cat.categories]
sq.pl.ligrec(sub, cluster_key='celltype', source_groups=groups, target_groups=groups,
             pvalue_threshold=0.05, remove_empty_interactions=True, title=' ')

> **Exercise.** Run neighbourhood enrichment on a different sample (e.g. an SLE core, X39)
and compare: are the same cell-type pairs co-located, or does the organisation differ by
disease?

In [ ]:
# your code here


> **Exercise.** The composition-based niches came from KMeans with k=6. Recompute them for
**k=4 and k=8** and compare each with k=6 using the Adjusted Rand Index. Does the major-
compartment structure stay stable, or does the number of niches change the story?

In [ ]:
# your code here


### Recap

Neighbourhood enrichment, niches (composition-based and BANKSY-style), and
ligand-receptor tests turn the cell map into tissue organisation and candidate signalling.
All are per-core and conditional on the graph; compare across samples before drawing disease
conclusions.